# Compute accessibility with cycling TTM
Zehui Yin

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
# read the data
analysis_grid = gpd.read_file('./output/analysis_grid_h3_res9.geojson')

ttm_weekday_offpeak = pd.read_parquet('./output/transit_ttm_weekday_off_peak_cycle.parquet')
ttm_weekday_peak = pd.read_parquet('./output/transit_ttm_weekday_peak_cycle.parquet')
ttm_weekend_offpeak = pd.read_parquet('./output/transit_ttm_weekend_off_peak_cycle.parquet')
ttm_weekend_peak = pd.read_parquet('./output/transit_ttm_weekend_peak_cycle.parquet')

In [ ]:
# fill na with 999999999
ttm_weekday_offpeak['travel_time'] = ttm_weekday_offpeak['travel_time'].fillna(999999999)
ttm_weekday_peak['travel_time'] = ttm_weekday_peak['travel_time'].fillna(999999999)
ttm_weekend_offpeak['travel_time'] = ttm_weekend_offpeak['travel_time'].fillna(999999999)
ttm_weekend_peak['travel_time'] = ttm_weekend_peak['travel_time'].fillna(999999999)

In [ ]:
# compute accessibility
def compute_accessibility(ttm_df, distance_decay_function):
    accessibility = (ttm_df
                     .groupby('from_id')['travel_time']
                     .apply(lambda x: (x.apply(distance_decay_function)).sum())
                     .reset_index()
                     )
    accessibility.columns = ['from_id', 'accessibility']
    return accessibility

In [ ]:
time_period_frames = {
    'weekday_off_peak': ttm_weekday_offpeak,
    'weekday_peak': ttm_weekday_peak,
    'weekend_off_peak': ttm_weekend_offpeak,
    'weekend_peak': ttm_weekend_peak,
}

accessibility_frames = []

for period_name, ttm_frame in time_period_frames.items():
    period_accessibility = compute_accessibility(
        ttm_frame,
        lambda x: (1 - (x / 60) ** 2) ** 2 if x <= 60 else 0,
    ).rename(columns={'accessibility': f'accessibility_{period_name}'})
    accessibility_frames.append(period_accessibility)

accessibility_df = accessibility_frames[0]
for period_accessibility in accessibility_frames[1:]:
    accessibility_df = accessibility_df.merge(
        period_accessibility,
        on='from_id',
        how='outer',
    )

accessibility_df.head()

In [ ]:
analysis_grid_with_accessibility = analysis_grid.merge(
    accessibility_df,
    left_on='h3_id',
    right_on='from_id',
    how='left',
)

analysis_grid_with_accessibility.head()

In [ ]:
analysis_grid_with_accessibility.explore(
    column='accessibility_weekend_peak',
    tooltip=['h3_id',
             'accessibility_weekday_peak',
             'accessibility_weekday_off_peak',
             'accessibility_weekend_peak',
             'accessibility_weekend_off_peak'], # type: ignore
)

In [ ]:
accessibility_columns = [
    'accessibility_weekday_off_peak',
    'accessibility_weekday_peak',
    'accessibility_weekend_off_peak',
    'accessibility_weekend_peak',
]

analysis_grid_with_accessibility[
    ['h3_id', *accessibility_columns]
].head()

In [ ]:
analysis_grid_with_accessibility.to_file(
    './output/analysis_grid_with_cycle_accessibility.geojson',
    driver='GeoJSON',
)